# RL-Based Algorithmic Trading Agent: Deep Q-Network on NIFTY 50

**Subproblem C** - Team K_Means_Kuch_Bhi

**Author:** Harjap Singh Bhatia

---

## Abstract

This notebook implements a Deep Q-Network (DQN) reinforcement learning agent for
algorithmic trading on the NIFTY 50 index constituents. The agent learns to output
Buy, Hold, or Sell decisions by maximising cumulative risk-adjusted returns through
trial-and-error interaction with a simulated trading environment built on historical
market data.

The project trains a single generic agent across all 50 NIFTY constituent stocks using
normalised feature representations, enabling transfer of learned trading patterns across
instruments. Performance is evaluated using Sharpe Ratio, Maximum Drawdown, and
Annualised Return, and compared against a passive Buy-and-Hold baseline.

## Project Context

This work constitutes Subproblem C of a four-part collaborative project:

| Subproblem | Owner | Deliverable |
|---|---|---|
| A | Nitesh Nunia | XGBoost supervised baseline (future comparison) |
| B | Myadarapu Adithyasai | Gymnasium trading environment |
| **C** | **Harjap Singh Bhatia** | **Trained DQN agent (this notebook)** |
| D | Dhairya Sharma | Sentiment pipeline and deployment |

## Theoretical Foundation

The agent learns Q-values satisfying the Bellman optimality equation:

    Q(s, a) = r + gamma * max_a' Q(s', a')

A neural network approximates Q(s, a; theta), trained by minimising:

    L(theta) = E[(r + gamma * max_a' Q(s', a'; theta_target) - Q(s, a; theta))^2]

Two stability mechanisms are employed: (1) experience replay buffer for decorrelating
sequential transitions, and (2) a periodically-synced target network to stabilise
the regression target.

## 1.1 Configuration

Set the dataset path below before running. All other configuration constants are
defined in this section and referenced throughout the notebook.

In [ ]:
# ===========================================================================
# USER CONFIGURATION - SET THIS BEFORE RUNNING
# ===========================================================================
# Point this to the directory containing the individual stock CSV files
# (e.g., RELIANCE.csv, TCS.csv, etc.) from the rohanrao/nifty50-stock-market-data
# Kaggle dataset.
#
# Example Kaggle path:
#   DATA_PATH = '/kaggle/input/nifty50-stock-market-data/'



# NOTE: USE GPU T4 FROM ACCELERATOR
# IT WILL TAKE AROUND 5-7 MIN FOR TESTING AND EVALUATING

DATA_PATH = '/kaggle/input/datasets/rohanrao/nifty50-stock-market-data'

print("Let's go!")

## 1.2 Dependencies

In [ ]:
import os
import glob
import time
import tracemalloc
import warnings
import json
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import BaseCallback

from sklearn.preprocessing import StandardScaler

import torch

warnings.filterwarnings('ignore')

print("All dependencies loaded successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1.3 Global Constants

In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

# ---------------------------------------------------------------------------
# Trading Environment Parameters
# ---------------------------------------------------------------------------
WINDOW_SIZE = 30              # Lookback window in trading days
NUM_ACTIONS = 3               # Buy=0, Hold=1, Sell=2
TRANSACTION_COST = 0.001      # 0.1% per trade
TRADING_DAYS_PER_YEAR = 252   # Standard for Indian/US equity markets
INITIAL_PORTFOLIO_VALUE = 1_000_000  # Starting capital in INR

# ---------------------------------------------------------------------------
# Data Split Ratios (chronological)
# ---------------------------------------------------------------------------
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# ---------------------------------------------------------------------------
# Visualisation Defaults
# ---------------------------------------------------------------------------
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.figsize': (12, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'serif',
})
sns.set_style("whitegrid")
PALETTE = sns.color_palette("muted", 10)

print("Global constants configured.")
print(f"  Window size: {WINDOW_SIZE}")
print(f"  Transaction cost: {TRANSACTION_COST*100:.1f}%")
print(f"  Data split: {TRAIN_RATIO:.0%} / {VAL_RATIO:.0%} / {TEST_RATIO:.0%}")

---

# 2. Data Ingestion

The dataset contains one CSV file per NIFTY 50 constituent stock, with daily
end-of-day trading data from the National Stock Exchange of India (NSE). Each file
follows a consistent 15-column schema covering price, volume, and delivery statistics.

We load all individual stock files and concatenate them into a single master DataFrame,
then perform basic validation of the schema and data integrity. The data has been
pre-cleaned and requires no imputation or transformation at this stage.

In [ ]:
# ---------------------------------------------------------------------------
# Load all stock CSV files
# ---------------------------------------------------------------------------
csv_files = sorted(glob.glob(os.path.join(DATA_PATH, '*.csv')))

# Exclude metadata and consolidated files
csv_files = [f for f in csv_files
             if 'metadata' not in os.path.basename(f).lower()
             and 'nifty50_all' not in os.path.basename(f).lower()
             and 'NIFTY50_all' not in os.path.basename(f)]

print(f"Found {len(csv_files)} stock CSV files in {DATA_PATH}")
print(f"First 5 files: {[os.path.basename(f) for f in csv_files[:5]]}")

# Load and concatenate
stock_frames = []
for fpath in csv_files:
    df_temp = pd.read_csv(fpath, parse_dates=['Date'])
    stock_frames.append(df_temp)

df_master = pd.concat(stock_frames, ignore_index=True)
df_master.sort_values(['Symbol', 'Date'], inplace=True)
df_master.reset_index(drop=True, inplace=True)

print(f"\nMaster DataFrame shape: {df_master.shape}")
print(f"Unique stocks: {df_master['Symbol'].nunique()}")
print(f"Date range: {df_master['Date'].min().date()} to {df_master['Date'].max().date()}")

In [ ]:
# ---------------------------------------------------------------------------
# Schema validation
# ---------------------------------------------------------------------------
EXPECTED_COLUMNS = [
    'Date', 'Symbol', 'Series', 'Prev Close', 'Open', 'High', 'Low',
    'Last', 'Close', 'VWAP', 'Volume', 'Turnover', 'Trades',
    'Deliverable Volume', '%Deliverble'
]

actual_cols = list(df_master.columns)
print("Column validation:")
for col in EXPECTED_COLUMNS:
    status = "[OK]" if col in actual_cols else "[MISSING]"
    print(f"  {status} {col}")

print(f"\nData types:\n{df_master.dtypes}")
print(f"\nMissing values per column:\n{df_master.isnull().sum()}")

In [ ]:
# ---------------------------------------------------------------------------
# Per-stock summary
# ---------------------------------------------------------------------------
stock_summary = df_master.groupby('Symbol').agg(
    start_date=('Date', 'min'),
    end_date=('Date', 'max'),
    num_days=('Date', 'count'),
    avg_close=('Close', 'mean'),
    avg_volume=('Volume', 'mean')
).round(2)

print(f"Per-stock summary (first 10 stocks):")
print(stock_summary.head(10).to_string())
print(f"\nTotal trading days across all stocks: {len(df_master):,}")

---

# 3. Exploratory Data Analysis (Pre-Feature Engineering)

Before engineering features, we examine the raw price and volume data to understand
market structure, identify regime changes, and assess data quality. The visualisations
below highlight the characteristics that the DQN agent will need to learn from.

In [ ]:
# ---------------------------------------------------------------------------
# 3.1 Closing price trajectories for representative stocks
# ---------------------------------------------------------------------------
representative_stocks = ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK']
# Filter to only stocks that exist in the dataset
representative_stocks = [s for s in representative_stocks
                         if s in df_master['Symbol'].unique()]

fig, ax = plt.subplots(figsize=(14, 6))
for i, symbol in enumerate(representative_stocks):
    stock_data = df_master[df_master['Symbol'] == symbol]
    ax.plot(stock_data['Date'], stock_data['Close'],
            label=symbol, color=PALETTE[i], linewidth=0.8)

ax.set_title('Closing Price Trajectories - Representative NIFTY 50 Stocks')
ax.set_xlabel('Date')
ax.set_ylabel('Close Price (INR)')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 3.2 Distribution of daily returns across all stocks
# ---------------------------------------------------------------------------
df_master['Daily_Return_Raw'] = df_master.groupby('Symbol')['Close'].pct_change()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_master['Daily_Return_Raw'].dropna(), bins=200, density=True,
             color=PALETTE[0], alpha=0.7, edgecolor='none')
axes[0].set_title('Distribution of Daily Returns (All Stocks)')
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Density')
axes[0].set_xlim(-0.15, 0.15)

# QQ-style: returns by year
yearly_stats = df_master.dropna(subset=['Daily_Return_Raw']).groupby(
    df_master['Date'].dt.year
)['Daily_Return_Raw'].agg(['mean', 'std'])

axes[1].bar(yearly_stats.index, yearly_stats['std'], color=PALETTE[1], alpha=0.8)
axes[1].set_title('Annual Volatility (Std Dev of Daily Returns)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Standard Deviation')

fig.tight_layout()
plt.show()

print(f"Overall daily return statistics:")
print(df_master['Daily_Return_Raw'].describe().to_string())

In [ ]:
# ---------------------------------------------------------------------------
# 3.3 Average daily volume by stock (top 20)
# ---------------------------------------------------------------------------
avg_volume = df_master.groupby('Symbol')['Volume'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
avg_volume.head(20).plot(kind='barh', ax=ax, color=PALETTE[2], edgecolor='none')
ax.set_title('Average Daily Trading Volume - Top 20 Stocks')
ax.set_xlabel('Average Volume (shares)')
ax.set_ylabel('Stock Symbol')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.invert_yaxis()
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 3.4 Correlation heatmap of raw OHLCV features (single stock)
# ---------------------------------------------------------------------------
sample_symbol = representative_stocks[0] if representative_stocks else df_master['Symbol'].unique()[0]
sample_data = df_master[df_master['Symbol'] == sample_symbol].copy()

raw_numeric_cols = ['Prev Close', 'Open', 'High', 'Low', 'Last', 'Close',
                    'VWAP', 'Volume', 'Turnover']
corr_matrix = sample_data[raw_numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title(f'Feature Correlation Matrix - {sample_symbol} (Raw OHLCV)')
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 3.5 Yearly return distribution boxplot
# ---------------------------------------------------------------------------
df_master['Year'] = df_master['Date'].dt.year
yearly_data = df_master.dropna(subset=['Daily_Return_Raw'])

fig, ax = plt.subplots(figsize=(14, 6))
years = sorted(yearly_data['Year'].unique())
data_by_year = [yearly_data[yearly_data['Year'] == y]['Daily_Return_Raw'].values
                for y in years]
bp = ax.boxplot(data_by_year, labels=years, patch_artist=True, showfliers=False,
                medianprops={'color': 'black', 'linewidth': 1.5})
for patch in bp['boxes']:
    patch.set_facecolor(PALETTE[3])
    patch.set_alpha(0.7)
ax.set_title('Distribution of Daily Returns by Year (All Stocks, Outliers Hidden)')
ax.set_xlabel('Year')
ax.set_ylabel('Daily Return')
plt.xticks(rotation=45)
fig.tight_layout()
plt.show()

### Key Observations from Exploratory Data Analysis

1. **Price scale heterogeneity.** Stock prices vary by orders of magnitude across the
   NIFTY 50 universe (e.g., single-digit to multi-thousand INR). This confirms the
   necessity of normalised/relative features rather than raw price levels for training
   a single generic agent.

2. **Volatility clustering.** Annual volatility spikes visibly during crisis periods
   (the 2008 global financial crisis, the 2020 COVID-19 crash), consistent with the
   well-documented phenomenon of volatility clustering in equity markets. The DQN
   agent must learn to adapt behaviour across calm and turbulent regimes.

3. **Volume concentration.** Trading volume is heavily concentrated in a handful of
   large-cap names. The agent will encounter very different liquidity environments
   across stocks, which the Volume Ratio feature (Section 4) is designed to capture.

4. **Return distribution.** Daily returns exhibit leptokurtosis (fat tails) and near-zero
   mean, consistent with stylised facts of financial returns. Extreme events are present
   in the data and will influence the agent's risk-adjusted performance metrics.

---

# 4. Feature Engineering

We compute 15 technical indicators from the raw OHLCV data. These features capture
momentum, volatility, trend, and microstructure signals that provide the agent with
a richer state representation than raw prices alone.

Each feature is computed per stock, respecting the time-series ordering. The warm-up
period required by the longest lookback window (SMA 50) means the first approximately
50 rows per stock will contain NaN values and will be dropped.

| # | Feature | Formula / Description |
|---|---|---|
| 1 | Log_Return | log(Close_t / Close_{t-1}) |
| 2 | Vol_10D | Rolling std of Log_Return, window=10 |
| 3 | Vol_20D | Rolling std of Log_Return, window=20 |
| 4 | Dist_SMA_10 | (Close - SMA_10) / SMA_10 |
| 5 | Dist_SMA_20 | (Close - SMA_20) / SMA_20 |
| 6 | Dist_SMA_50 | (Close - SMA_50) / SMA_50 |
| 7 | RSI_14 | Relative Strength Index, 14-day window |
| 8 | MACD | EMA_12 - EMA_26 |
| 9 | MACD_Signal | EMA_9 of MACD |
| 10 | MACD_Diff | MACD - MACD_Signal |
| 11 | BB_Pband | Bollinger Band %B (20-day, 2 std) |
| 12 | ATR_14 | Average True Range, 14-day window |
| 13 | Volume_Ratio_20 | Volume / SMA_20(Volume) |
| 14 | VWAP_Dist | (Close - VWAP) / VWAP |
| 15 | Deliverable_Pct | Deliverable Volume / Volume |

In [ ]:
def compute_rsi(series, window=14):
    """Compute Relative Strength Index."""
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.rolling(window=window, min_periods=window).mean()
    avg_loss = loss.rolling(window=window, min_periods=window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi


def compute_engineered_features(df):
    """
    Compute all 15 engineered features for a single stock DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame for one stock, sorted by Date ascending.

    Returns
    -------
    pd.DataFrame
        Input DataFrame with 15 new feature columns appended.
    """
    df = df.copy()

    # 1. Log Return
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

    # 2-3. Rolling Volatility
    df['Vol_10D'] = df['Log_Return'].rolling(window=10).std()
    df['Vol_20D'] = df['Log_Return'].rolling(window=20).std()

    # 4-6. Distance from Simple Moving Averages
    for w in [10, 20, 50]:
        sma = df['Close'].rolling(window=w).mean()
        df[f'Dist_SMA_{w}'] = (df['Close'] - sma) / sma

    # 7. RSI
    df['RSI_14'] = compute_rsi(df['Close'], window=14)

    # 8-10. MACD
    ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema_12 - ema_26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Diff'] = df['MACD'] - df['MACD_Signal']

    # 11. Bollinger Band %B
    sma_20 = df['Close'].rolling(window=20).mean()
    std_20 = df['Close'].rolling(window=20).std()
    upper_band = sma_20 + 2 * std_20
    lower_band = sma_20 - 2 * std_20
    df['BB_Pband'] = (df['Close'] - lower_band) / (upper_band - lower_band)

    # 12. Average True Range
    high_low = df['High'] - df['Low']
    high_close_prev = (df['High'] - df['Close'].shift(1)).abs()
    low_close_prev = (df['Low'] - df['Close'].shift(1)).abs()
    true_range = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
    df['ATR_14'] = true_range.rolling(window=14).mean()

    # 13. Volume Ratio
    vol_sma_20 = df['Volume'].rolling(window=20).mean()
    df['Volume_Ratio_20'] = df['Volume'] / vol_sma_20.replace(0, np.nan)

    # 14. VWAP Distance
    df['VWAP_Dist'] = (df['Close'] - df['VWAP']) / df['VWAP'].replace(0, np.nan)

    # 15. Deliverable Percentage
    # Use the existing %Deliverble column if available, otherwise compute
    if '%Deliverble' in df.columns:
        df['Deliverable_Pct'] = df['%Deliverble']
    else:
        df['Deliverable_Pct'] = df['Deliverable Volume'] / df['Volume'].replace(0, np.nan)

    return df


print("Feature engineering functions defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Apply feature engineering to all stocks
# ---------------------------------------------------------------------------
engineered_frames = []
symbols = df_master['Symbol'].unique()

for symbol in symbols:
    stock_df = df_master[df_master['Symbol'] == symbol].copy()
    stock_df = stock_df.sort_values('Date').reset_index(drop=True)
    stock_df = compute_engineered_features(stock_df)
    engineered_frames.append(stock_df)

df_engineered = pd.concat(engineered_frames, ignore_index=True)

# Drop rows with NaN from warm-up period
initial_rows = len(df_engineered)
df_engineered.dropna(subset=[
    'Log_Return', 'Vol_10D', 'Vol_20D', 'Dist_SMA_10', 'Dist_SMA_20',
    'Dist_SMA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Diff',
    'BB_Pband', 'ATR_14', 'Volume_Ratio_20', 'VWAP_Dist', 'Deliverable_Pct'
], inplace=True)
df_engineered.reset_index(drop=True, inplace=True)

dropped_rows = initial_rows - len(df_engineered)
print(f"Rows before dropping NaN (warm-up): {initial_rows:,}")
print(f"Rows dropped: {dropped_rows:,}")
print(f"Rows after feature engineering: {len(df_engineered):,}")
print(f"\nNew columns added:")
eng_features = ['Log_Return', 'Vol_10D', 'Vol_20D', 'Dist_SMA_10', 'Dist_SMA_20',
                'Dist_SMA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Diff',
                'BB_Pband', 'ATR_14', 'Volume_Ratio_20', 'VWAP_Dist', 'Deliverable_Pct']
for feat in eng_features:
    print(f"  {feat}: mean={df_engineered[feat].mean():.4f}, std={df_engineered[feat].std():.4f}")

---

# 5. Sentiment Feature Integration (Placeholder)

The full state vector specification (features.txt) includes four sentiment features
derived from a financial news pipeline (Subproblem D):

| Feature | Description |
|---|---|
| Sentiment_Score | Aggregate sentiment polarity from news articles |
| Sentiment_Magnitude | Strength/confidence of the sentiment signal |
| Article_Count | Number of relevant news articles per day |
| Sentiment_Rolling_3D | 3-day rolling average of Sentiment_Score |

These features are not yet available and will be integrated once Subproblem D delivers
the sentiment pipeline. The current model uses 27 numeric features (12 raw + 15
engineered). When sentiment data becomes available, the state vector will expand to 31
features, and the environment's observation space will be updated accordingly.

No code changes are required at this stage. The environment and model architecture are
designed to accept a variable number of features, controlled by the feature column list
defined in Section 7.

---

# 6. Post-Feature Engineering Visualisation

We now examine the distributions and relationships of the engineered features to
verify that they are well-behaved (finite, reasonably scaled) and that they capture
distinct aspects of market behaviour.

In [ ]:
# ---------------------------------------------------------------------------
# 6.1 Distribution of key engineered features
# ---------------------------------------------------------------------------
key_features = ['RSI_14', 'MACD_Diff', 'Vol_10D', 'BB_Pband']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat, color in zip(axes.flat, key_features, PALETTE[:4]):
    data = df_engineered[feat].dropna()
    # Clip extreme outliers for visualisation
    lower, upper = data.quantile(0.01), data.quantile(0.99)
    clipped = data[(data >= lower) & (data <= upper)]
    ax.hist(clipped, bins=100, density=True, color=color, alpha=0.7, edgecolor='none')
    ax.set_title(f'Distribution of {feat}')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.axvline(clipped.mean(), color='black', linestyle='--', linewidth=1, label='Mean')
    ax.legend()

fig.suptitle('Engineered Feature Distributions (1st-99th Percentile)', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 6.2 Correlation matrix of all model features
# ---------------------------------------------------------------------------
RAW_FEATURE_COLS = ['Prev Close', 'Open', 'High', 'Low', 'Last', 'Close',
                    'VWAP', 'Volume', 'Turnover', 'Trades',
                    'Deliverable Volume', 'Deliverable_Pct']

ENG_FEATURE_COLS = ['Log_Return', 'Vol_10D', 'Vol_20D', 'Dist_SMA_10',
                    'Dist_SMA_20', 'Dist_SMA_50', 'RSI_14', 'MACD',
                    'MACD_Signal', 'MACD_Diff', 'BB_Pband', 'ATR_14',
                    'Volume_Ratio_20', 'VWAP_Dist']

# Use a subset of features for readability
vis_features = ENG_FEATURE_COLS + ['Close', 'Volume']
corr_eng = df_engineered[vis_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_eng, dtype=bool), k=1)
sns.heatmap(corr_eng, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.7})
ax.set_title('Correlation Matrix - Engineered Features')
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 6.3 Feature correlation with next-day returns
# ---------------------------------------------------------------------------
df_engineered['Next_Day_Return'] = df_engineered.groupby('Symbol')['Log_Return'].shift(-1)

all_feature_cols = ENG_FEATURE_COLS.copy()
corr_with_target = df_engineered[all_feature_cols + ['Next_Day_Return']].corr()['Next_Day_Return'].drop('Next_Day_Return')
corr_with_target = corr_with_target.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = [PALETTE[0] if v >= 0 else PALETTE[3] for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='none')
ax.set_title('Feature Correlation with Next-Day Log Return')
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='black', linewidth=0.8)
fig.tight_layout()
plt.show()

# Clean up temporary column
df_engineered.drop('Next_Day_Return', axis=1, inplace=True, errors='ignore')

### Key Observations from Engineered Features

1. **RSI distribution.** RSI_14 clusters around 40-60, with tails at the canonical
   overbought (>70) and oversold (<30) thresholds. This suggests the feature has
   discriminative power for identifying extreme sentiment regimes.

2. **MACD and momentum.** MACD_Diff (the histogram component) is approximately symmetric
   around zero, as expected. Its correlation with next-day returns provides a weak but
   directionally useful signal.

3. **Volatility features.** Vol_10D and Vol_20D are right-skewed (volatility is bounded
   below by zero) and exhibit the expected persistence. These will help the agent
   calibrate position sizing during high-volatility regimes.

4. **Feature redundancy.** The correlation matrix reveals expected collinearity among
   price-level features (Open, High, Low, Close, VWAP) and among the MACD family. The
   agent's neural network should learn to exploit these correlations without explicit
   feature selection, but awareness of this structure is important for interpretation.

---

# 7. Data Preparation for Reinforcement Learning

This section prepares the data for the trading environment by:
1. Defining the feature columns that constitute the state vector.
2. Splitting data chronologically per stock into train/validation/test sets.
3. Fitting a z-score normaliser (StandardScaler) on the training data only, then
   transforming all three splits to prevent lookahead bias.

In [ ]:
# ---------------------------------------------------------------------------
# 7.1 Define the state feature columns
# ---------------------------------------------------------------------------
# These are the numeric columns that form the per-timestep state vector.
# Date, Symbol, Series, and other metadata are excluded.
STATE_FEATURES = [
    # Raw numeric features (12)
    'Prev Close', 'Open', 'High', 'Low', 'Last', 'Close',
    'VWAP', 'Volume', 'Turnover', 'Trades',
    'Deliverable Volume', 'Deliverable_Pct',
    # Engineered features (15)
    'Log_Return', 'Vol_10D', 'Vol_20D',
    'Dist_SMA_10', 'Dist_SMA_20', 'Dist_SMA_50',
    'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Diff',
    'BB_Pband', 'ATR_14', 'Volume_Ratio_20', 'VWAP_Dist',
    # Deliverable_Pct already counted in raw
]

# Remove any duplicates while preserving order
seen = set()
STATE_FEATURES = [x for x in STATE_FEATURES if not (x in seen or seen.add(x))]

# Verify all columns exist
missing = [c for c in STATE_FEATURES if c not in df_engineered.columns]
if missing:
    print(f"[WARNING] Missing columns: {missing}")
    STATE_FEATURES = [c for c in STATE_FEATURES if c in df_engineered.columns]

NUM_FEATURES = len(STATE_FEATURES)
OBS_DIM = WINDOW_SIZE * NUM_FEATURES + 1  # +1 for position

print(f"State features ({NUM_FEATURES} columns):")
for i, f in enumerate(STATE_FEATURES, 1):
    print(f"  {i:2d}. {f}")
print(f"\nObservation dimension: {WINDOW_SIZE} x {NUM_FEATURES} + 1 = {OBS_DIM}")

In [ ]:
# ---------------------------------------------------------------------------
# 7.2 Chronological train/val/test split per stock
# ---------------------------------------------------------------------------
train_frames = []
val_frames = []
test_frames = []

for symbol in df_engineered['Symbol'].unique():
    stock_df = df_engineered[df_engineered['Symbol'] == symbol].copy()
    stock_df = stock_df.sort_values('Date').reset_index(drop=True)

    n = len(stock_df)
    train_end = int(n * TRAIN_RATIO)
    val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

    train_frames.append(stock_df.iloc[:train_end])
    val_frames.append(stock_df.iloc[train_end:val_end])
    test_frames.append(stock_df.iloc[val_end:])

df_train = pd.concat(train_frames, ignore_index=True)
df_val = pd.concat(val_frames, ignore_index=True)
df_test = pd.concat(test_frames, ignore_index=True)

print(f"Data split complete:")
print(f"  Training set:   {len(df_train):>8,} rows ({len(df_train)/len(df_engineered):.1%})")
print(f"  Validation set: {len(df_val):>8,} rows ({len(df_val)/len(df_engineered):.1%})")
print(f"  Test set:       {len(df_test):>8,} rows ({len(df_test)/len(df_engineered):.1%})")

# Date ranges per split
for name, split_df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f"  {name} date range: {split_df['Date'].min().date()} to {split_df['Date'].max().date()}")

In [ ]:
# ---------------------------------------------------------------------------
# 7.3 Z-score normalisation (fit on training data only)
# ---------------------------------------------------------------------------
scaler = StandardScaler()

# Replace any remaining inf values with NaN, then fill with column mean
for split_df in [df_train, df_val, df_test]:
    split_df[STATE_FEATURES] = split_df[STATE_FEATURES].replace([np.inf, -np.inf], np.nan)

# Fit scaler on training data
train_feature_values = df_train[STATE_FEATURES].fillna(df_train[STATE_FEATURES].mean())
scaler.fit(train_feature_values)

# Transform all splits
df_train_scaled = df_train.copy()
df_val_scaled = df_val.copy()
df_test_scaled = df_test.copy()

for split_df in [df_train_scaled, df_val_scaled, df_test_scaled]:
    filled = split_df[STATE_FEATURES].fillna(train_feature_values.mean())
    split_df[STATE_FEATURES] = scaler.transform(filled)

print("Z-score normalisation applied (fit on training data only).")
print(f"\nTraining set feature statistics (post-normalisation):")
print(df_train_scaled[STATE_FEATURES].describe().loc[['mean', 'std']].round(4).to_string())

---

# 8. Trading Environment

The trading environment implements the Gymnasium interface required by Stable-Baselines3.
Two environment classes are defined:

1. **NiftyTradingEnv** - operates on a single stock's data. Used for evaluation.
2. **MultiStockTradingEnv** - wraps multiple stocks and randomly selects one per episode.
   Used for training the generic cross-stock agent.

The environment contract follows Section 6 of the project context:
- **State:** 30-day rolling window of normalised features + current position
- **Action:** Discrete(3) - Buy(0), Hold(1), Sell(2)
- **Reward:** r_t = daily_return * position - transaction_cost * |delta_position|

In [ ]:
class NiftyTradingEnv(gym.Env):
    """
    Gymnasium-compatible trading environment for a single stock.

    The agent observes a rolling window of normalised features and a binary
    position indicator, then selects Buy/Hold/Sell. The reward is the
    position-weighted daily return minus transaction costs on trades.
    """

    metadata = {"render_modes": []}

    def __init__(self, stock_data, feature_cols, window_size=WINDOW_SIZE,
                 transaction_cost=TRANSACTION_COST, random_start=True):
        """
        Parameters
        ----------
        stock_data : pd.DataFrame
            DataFrame for one stock with normalised feature columns.
        feature_cols : list of str
            Column names to include in the observation.
        window_size : int
            Number of historical days in the observation window.
        transaction_cost : float
            Proportional transaction cost per trade.
        random_start : bool
            If True, randomly sample a starting index each episode (training).
            If False, always start from the beginning (evaluation).
        """
        super().__init__()

        self.feature_cols = feature_cols
        self.window_size = window_size
        self.transaction_cost = transaction_cost
        self.random_start = random_start

        # Extract feature matrix and close prices
        self._features = stock_data[feature_cols].values.astype(np.float32)
        self._close_prices = stock_data['Close'].values  # un-normalised for reward calc
        # For reward computation, we need the original (un-scaled) close column.
        # But since we scaled everything, use the raw pct change from the original data.
        # Store the original log returns for reward.
        if 'Log_Return' in stock_data.columns:
            # Log_Return column exists but is normalised. Use raw close pct_change instead.
            pass
        # We store the raw close prices from the pre-scaled data for reward calculation.
        # Actually, we need to pass raw closes separately or recompute returns.
        # For simplicity, compute returns from the raw close column stored before scaling.
        self._raw_closes = stock_data['Close'].values.copy()

        self.n_steps = len(self._features)
        num_features = len(feature_cols)

        # Spaces
        obs_dim = window_size * num_features + 1  # +1 for position
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(NUM_ACTIONS)

        # Internal state
        self._position = 0
        self._current_step = 0
        self._start_step = 0
        self._portfolio_value = INITIAL_PORTFOLIO_VALUE
        self._entry_price = 0.0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)

        self._position = 0
        self._portfolio_value = INITIAL_PORTFOLIO_VALUE
        self._entry_price = 0.0

        if self.random_start:
            max_start = self.n_steps - self.window_size - TRADING_DAYS_PER_YEAR
            if max_start <= self.window_size:
                self._start_step = self.window_size
            else:
                self._start_step = self.np_random.integers(
                    self.window_size, max_start + 1
                )
        else:
            self._start_step = self.window_size

        self._current_step = self._start_step
        obs = self._get_observation()
        return obs, {}

    def step(self, action):
        old_position = self._position

        # Determine new position
        if action == 0:    # Buy
            new_position = 1
        elif action == 2:  # Sell
            new_position = 0
        else:              # Hold
            new_position = old_position

        position_changed = abs(new_position - old_position)

        # Compute reward
        if self._current_step < self.n_steps:
            curr_close = self._raw_closes[self._current_step]
            prev_close = self._raw_closes[self._current_step - 1]
            if prev_close != 0:
                daily_return = (curr_close - prev_close) / prev_close
            else:
                daily_return = 0.0
        else:
            daily_return = 0.0

        reward = float(
            daily_return * new_position
            - self.transaction_cost * position_changed
        )

        # Update portfolio value
        self._portfolio_value *= (1 + daily_return * new_position
                                  - self.transaction_cost * position_changed)

        # Advance
        self._position = new_position
        self._current_step += 1

        terminated = False
        truncated = self._current_step >= self.n_steps - 1

        obs = self._get_observation()
        info = {
            "position": self._position,
            "daily_return": daily_return,
            "portfolio_value": self._portfolio_value,
        }

        return obs, reward, terminated, truncated, info

    def _get_observation(self):
        start = max(0, self._current_step - self.window_size)
        end = self._current_step

        if end > self.n_steps:
            end = self.n_steps
        if end - start < self.window_size:
            # Pad with zeros if not enough history
            window = np.zeros((self.window_size, len(self.feature_cols)), dtype=np.float32)
            available = self._features[start:end]
            window[-len(available):] = available
        else:
            window = self._features[start:end]

        flat = window.flatten()
        position_feature = np.array([self._position], dtype=np.float32)
        obs = np.concatenate([flat, position_feature])
        return obs


print("NiftyTradingEnv defined.")

In [ ]:
class MultiStockTradingEnv(gym.Env):
    """
    Wraps multiple NiftyTradingEnv instances, randomly selecting a stock on
    each episode reset. This enables training a single generic agent across
    the full NIFTY 50 universe.
    """

    metadata = {"render_modes": []}

    def __init__(self, stock_data_dict, feature_cols, window_size=WINDOW_SIZE,
                 transaction_cost=TRANSACTION_COST):
        """
        Parameters
        ----------
        stock_data_dict : dict[str, pd.DataFrame]
            Mapping from stock symbol to its normalised DataFrame.
        feature_cols : list of str
            Column names for the observation.
        """
        super().__init__()

        self.feature_cols = feature_cols
        self.window_size = window_size
        self.transaction_cost = transaction_cost

        # Create per-stock environments
        self.stock_symbols = list(stock_data_dict.keys())
        self.stock_envs = {}
        for symbol, data in stock_data_dict.items():
            if len(data) > window_size + 10:  # Need enough data for at least a short episode
                self.stock_envs[symbol] = NiftyTradingEnv(
                    stock_data=data,
                    feature_cols=feature_cols,
                    window_size=window_size,
                    transaction_cost=transaction_cost,
                    random_start=True,
                )

        self.stock_symbols = list(self.stock_envs.keys())
        assert len(self.stock_symbols) > 0, "No stocks with sufficient data."

        # Copy spaces from any sub-environment
        ref_env = self.stock_envs[self.stock_symbols[0]]
        self.observation_space = ref_env.observation_space
        self.action_space = ref_env.action_space

        self._current_env = None
        self._current_symbol = None

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)

        # Randomly select a stock
        idx = self.np_random.integers(0, len(self.stock_symbols))
        self._current_symbol = self.stock_symbols[idx]
        self._current_env = self.stock_envs[self._current_symbol]

        obs, info = self._current_env.reset(seed=seed)
        info['symbol'] = self._current_symbol
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self._current_env.step(action)
        info['symbol'] = self._current_symbol
        return obs, reward, terminated, truncated, info


print("MultiStockTradingEnv defined.")

In [ ]:
# ---------------------------------------------------------------------------
# 8.1 Prepare stock data dictionaries for each split
# ---------------------------------------------------------------------------
def build_stock_dict(df, feature_cols):
    """Build a {symbol: DataFrame} dict from a concatenated DataFrame."""
    stock_dict = {}
    for symbol in df['Symbol'].unique():
        stock_df = df[df['Symbol'] == symbol].sort_values('Date').reset_index(drop=True)
        if len(stock_df) > WINDOW_SIZE + 10:
            stock_dict[symbol] = stock_df
    return stock_dict

# We need the raw close prices for reward calculation.
# Since df_train_scaled has normalised Close, we need to store raw closes.
# Rebuild stock dicts with raw close prices available.

# For the environment, we need normalised features but raw closes for rewards.
# Strategy: add a 'Raw_Close' column to the scaled DataFrames before building envs.

for split_name, scaled_df, raw_df in [
    ('train', df_train_scaled, df_train),
    ('val', df_val_scaled, df_val),
    ('test', df_test_scaled, df_test)
]:
    scaled_df['Raw_Close'] = raw_df['Close'].values

# Update the NiftyTradingEnv to use Raw_Close for reward calculation
# by re-assigning _raw_closes after construction.

train_stock_dict = build_stock_dict(df_train_scaled, STATE_FEATURES)
val_stock_dict = build_stock_dict(df_val_scaled, STATE_FEATURES)
test_stock_dict = build_stock_dict(df_test_scaled, STATE_FEATURES)

print(f"Stocks with sufficient data:")
print(f"  Training:   {len(train_stock_dict)} stocks")
print(f"  Validation: {len(val_stock_dict)} stocks")
print(f"  Test:       {len(test_stock_dict)} stocks")

In [ ]:
# ---------------------------------------------------------------------------
# 8.2 Fix raw close prices in environments
# ---------------------------------------------------------------------------
# The NiftyTradingEnv uses self._raw_closes for reward computation.
# Since we normalised the Close column, we need to patch in the original prices.

class NiftyTradingEnvFixed(NiftyTradingEnv):
    """NiftyTradingEnv with correct raw close prices for reward calculation."""

    def __init__(self, stock_data, feature_cols, **kwargs):
        super().__init__(stock_data, feature_cols, **kwargs)
        # Override raw closes with the actual un-normalised close prices
        if 'Raw_Close' in stock_data.columns:
            self._raw_closes = stock_data['Raw_Close'].values.astype(np.float64)


class MultiStockTradingEnvFixed(MultiStockTradingEnv):
    """MultiStockTradingEnv using NiftyTradingEnvFixed sub-environments."""

    def __init__(self, stock_data_dict, feature_cols, **kwargs):
        # Override parent init to use Fixed env
        gym.Env.__init__(self)
        self.feature_cols = feature_cols
        self.window_size = kwargs.get('window_size', WINDOW_SIZE)
        self.transaction_cost = kwargs.get('transaction_cost', TRANSACTION_COST)

        self.stock_symbols = list(stock_data_dict.keys())
        self.stock_envs = {}
        for symbol, data in stock_data_dict.items():
            if len(data) > self.window_size + 10:
                self.stock_envs[symbol] = NiftyTradingEnvFixed(
                    stock_data=data,
                    feature_cols=feature_cols,
                    window_size=self.window_size,
                    transaction_cost=self.transaction_cost,
                    random_start=True,
                )

        self.stock_symbols = list(self.stock_envs.keys())
        assert len(self.stock_symbols) > 0, "No stocks with sufficient data."

        ref_env = self.stock_envs[self.stock_symbols[0]]
        self.observation_space = ref_env.observation_space
        self.action_space = ref_env.action_space
        self._current_env = None
        self._current_symbol = None


print("Fixed environment classes defined.")

In [ ]:
# ---------------------------------------------------------------------------
# 8.3 Validate environment with SB3 check_env
# ---------------------------------------------------------------------------
sample_symbol = list(train_stock_dict.keys())[0]
sample_env = NiftyTradingEnvFixed(
    stock_data=train_stock_dict[sample_symbol],
    feature_cols=STATE_FEATURES,
    random_start=False,
)

try:
    check_env(sample_env, warn=True)
    print(f"[PASS] Environment passed SB3 check_env validation.")
except Exception as e:
    print(f"[FAIL] Environment validation error: {e}")

print(f"  Observation space: {sample_env.observation_space}")
print(f"  Action space: {sample_env.action_space}")
print(f"  Observation dim: {sample_env.observation_space.shape[0]}")

# Quick step test
obs, info = sample_env.reset()
print(f"  Reset observation shape: {obs.shape}")
obs, reward, term, trunc, info = sample_env.step(0)
print(f"  Step observation shape: {obs.shape}, reward: {reward:.6f}")
del sample_env

---

# 9. Hyperparameter Optimisation

Rather than hardcoding hyperparameter values, we perform a randomised search over
the most impactful DQN parameters. Each candidate configuration is trained for a
short run, then evaluated on the validation set using the Sharpe Ratio as the
selection criterion.

The search space covers:
- **Learning rate (alpha):** controls gradient step size
- **Discount factor (gamma):** controls the agent's time horizon
- **Exploration fraction:** portion of training devoted to epsilon decay
- **Batch size:** number of transitions sampled per gradient update

To keep computation feasible within Kaggle resource limits, we use a small number
of training timesteps per candidate and sample a subset of configurations from the
full grid.

In [ ]:
# ---------------------------------------------------------------------------
# 9.1 Evaluation helper for hyperparameter search
# ---------------------------------------------------------------------------
def evaluate_agent_on_validation(model, val_stock_dict, feature_cols, n_episodes=5):
    """
    Run the agent on a random sample of validation stocks and return
    the average Sharpe Ratio.
    """
    sharpe_ratios = []

    symbols = list(val_stock_dict.keys())
    if len(symbols) == 0:
        return 0.0

    sample_symbols = np.random.choice(symbols, size=min(n_episodes, len(symbols)), replace=False)

    for symbol in sample_symbols:
        env = NiftyTradingEnvFixed(
            stock_data=val_stock_dict[symbol],
            feature_cols=feature_cols,
            random_start=False,
        )

        obs, _ = env.reset()
        daily_returns = []
        done = False

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))
            daily_returns.append(info.get('daily_return', 0.0) * info.get('position', 0))
            done = terminated or truncated

        if len(daily_returns) > 1:
            daily_returns = np.array(daily_returns)
            std = np.std(daily_returns, ddof=1)
            if std > 0:
                sharpe = (np.mean(daily_returns) / std) * np.sqrt(TRADING_DAYS_PER_YEAR)
            else:
                sharpe = 0.0
            sharpe_ratios.append(sharpe)

    return np.mean(sharpe_ratios) if sharpe_ratios else 0.0


print("Validation evaluation function defined.")

In [ ]:
# ---------------------------------------------------------------------------
# 9.2 Hyperparameter search
# ---------------------------------------------------------------------------
HP_SEARCH_TIMESTEPS = 10_000  # Short training per candidate

param_grid = {
    'learning_rate': [1e-5, 5e-5, 1e-4, 5e-4, 1e-3],
    'gamma': [0.95, 0.97, 0.99],
    'exploration_fraction': [0.1, 0.2, 0.3],
    'batch_size': [32, 64, 128],
}

# Total combinations
total_combos = 1
for v in param_grid.values():
    total_combos *= len(v)
print(f"Total grid combinations: {total_combos}")

# Random sample of configurations
N_SEARCH_CANDIDATES = min(15, total_combos)
np.random.seed(RANDOM_SEED)

candidates = []
for _ in range(N_SEARCH_CANDIDATES):
    config = {k: np.random.choice(v) for k, v in param_grid.items()}
    # Ensure numeric types
    config['learning_rate'] = float(config['learning_rate'])
    config['gamma'] = float(config['gamma'])
    config['exploration_fraction'] = float(config['exploration_fraction'])
    config['batch_size'] = int(config['batch_size'])
    candidates.append(config)

print(f"Evaluating {N_SEARCH_CANDIDATES} random configurations...")
print("-" * 80)

results = []
train_env = MultiStockTradingEnvFixed(train_stock_dict, STATE_FEATURES)

for i, config in enumerate(candidates):
    print(f"  Candidate {i+1}/{N_SEARCH_CANDIDATES}: lr={config['learning_rate']:.1e}, "
          f"gamma={config['gamma']:.2f}, explore={config['exploration_fraction']:.1f}, "
          f"batch={config['batch_size']}")

    try:
        model = DQN(
            policy="MlpPolicy",
            env=train_env,
            learning_rate=config['learning_rate'],
            gamma=config['gamma'],
            buffer_size=50_000,
            batch_size=config['batch_size'],
            exploration_fraction=config['exploration_fraction'],
            exploration_final_eps=0.05,
            verbose=0,
            seed=RANDOM_SEED,
        )
        model.learn(total_timesteps=HP_SEARCH_TIMESTEPS)

        val_sharpe = evaluate_agent_on_validation(model, val_stock_dict, STATE_FEATURES)
        config['val_sharpe'] = val_sharpe
        results.append(config)
        print(f"    -> Validation Sharpe: {val_sharpe:.4f}")

        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    except Exception as e:
        print(f"    -> Failed: {e}")
        config['val_sharpe'] = -999.0
        results.append(config)

print("-" * 80)

In [ ]:
# ---------------------------------------------------------------------------
# 9.3 Select best hyperparameters
# ---------------------------------------------------------------------------
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('val_sharpe', ascending=False)

print("Hyperparameter Search Results (sorted by validation Sharpe Ratio):")
print(results_df.to_string(index=False))

best_config = results_df.iloc[0].to_dict()
best_config.pop('val_sharpe')

# Ensure correct types
BEST_LR = float(best_config['learning_rate'])
BEST_GAMMA = float(best_config['gamma'])
BEST_EXPLORE_FRAC = float(best_config['exploration_fraction'])
BEST_BATCH_SIZE = int(best_config['batch_size'])

print(f"\nBest configuration selected:")
print(f"  Learning rate:        {BEST_LR:.1e}")
print(f"  Gamma:                {BEST_GAMMA:.2f}")
print(f"  Exploration fraction: {BEST_EXPLORE_FRAC:.1f}")
print(f"  Batch size:           {BEST_BATCH_SIZE}")

---

# 10. Model Training

With the optimal hyperparameters identified, we now train the DQN agent for a full
run across all 50 stocks. Training uses the MultiStockTradingEnv, which randomly
samples a stock on each episode reset, producing a generic agent.

We log episode rewards via a custom callback for plotting the training curve.
Wall-clock time and peak memory consumption are measured for the complete training
phase.

In [ ]:
# ---------------------------------------------------------------------------
# 10.1 Training callback for reward logging
# ---------------------------------------------------------------------------
class RewardLoggerCallback(BaseCallback):
    """
    Logs per-episode cumulative reward during training.
    """

    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []
        self._current_reward = 0.0
        self._current_length = 0

    def _on_step(self) -> bool:
        self._current_reward += self.locals['rewards'][0]
        self._current_length += 1

        # Check if episode ended
        if self.locals['dones'][0]:
            self.episode_rewards.append(self._current_reward)
            self.episode_lengths.append(self._current_length)
            self._current_reward = 0.0
            self._current_length = 0

        return True


print("RewardLoggerCallback defined.")

In [ ]:
# ---------------------------------------------------------------------------
# 10.2 Full training run
# ---------------------------------------------------------------------------
TRAIN_TIMESTEPS = 100_000  # Adjust based on available compute

train_env_full = MultiStockTradingEnvFixed(train_stock_dict, STATE_FEATURES)

print("=" * 70)
print(f"Starting DQN training for {TRAIN_TIMESTEPS:,} timesteps")
print(f"  Stocks in training pool: {len(train_stock_dict)}")
print(f"  Hyperparameters: lr={BEST_LR:.1e}, gamma={BEST_GAMMA}, "
      f"batch={BEST_BATCH_SIZE}, explore={BEST_EXPLORE_FRAC}")
print("=" * 70)

# Start performance measurement
tracemalloc.start()
train_start_time = time.perf_counter()
train_mem_snapshot_before = tracemalloc.take_snapshot()

callback = RewardLoggerCallback()

model = DQN(
    policy="MlpPolicy",
    env=train_env_full,
    learning_rate=BEST_LR,
    gamma=BEST_GAMMA,
    buffer_size=100_000,
    batch_size=BEST_BATCH_SIZE,
    exploration_fraction=BEST_EXPLORE_FRAC,
    exploration_final_eps=0.05,
    target_update_interval=1000,
    train_freq=4,
    gradient_steps=1,
    verbose=1,
    seed=RANDOM_SEED,
)

model.learn(total_timesteps=TRAIN_TIMESTEPS, callback=callback)

# Stop performance measurement
train_end_time = time.perf_counter()
train_mem_snapshot_after = tracemalloc.take_snapshot()
train_wall_time = train_end_time - train_start_time

# Calculate memory delta
train_mem_stats = train_mem_snapshot_after.compare_to(train_mem_snapshot_before, 'lineno')
train_peak_memory_bytes = sum(stat.size_diff for stat in train_mem_stats if stat.size_diff > 0)
train_peak_memory_mb = train_peak_memory_bytes / (1024 * 1024)

tracemalloc.stop()

print("\n" + "=" * 70)
print(f"Training complete.")
print(f"  Wall-clock time: {train_wall_time:.2f} seconds ({train_wall_time/60:.1f} minutes)")
print(f"  Memory delta: {train_peak_memory_mb:.2f} MB")
print(f"  Episodes completed: {len(callback.episode_rewards)}")
print("=" * 70)

In [ ]:
# ---------------------------------------------------------------------------
# 10.3 Plot training curves
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Episode reward
rewards = np.array(callback.episode_rewards)
window = max(1, len(rewards) // 50)  # Adaptive smoothing window
if len(rewards) > window:
    smoothed = pd.Series(rewards).rolling(window=window, min_periods=1).mean().values
else:
    smoothed = rewards

axes[0].plot(rewards, alpha=0.3, color=PALETTE[0], linewidth=0.5, label='Raw')
axes[0].plot(smoothed, color=PALETTE[0], linewidth=2, label=f'Smoothed (window={window})')
axes[0].set_title('Episode Reward During Training')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Cumulative Episode Reward')
axes[0].legend()
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)

# Episode length
lengths = np.array(callback.episode_lengths)
if len(lengths) > window:
    smoothed_len = pd.Series(lengths).rolling(window=window, min_periods=1).mean().values
else:
    smoothed_len = lengths

axes[1].plot(lengths, alpha=0.3, color=PALETTE[1], linewidth=0.5, label='Raw')
axes[1].plot(smoothed_len, color=PALETTE[1], linewidth=2, label=f'Smoothed (window={window})')
axes[1].set_title('Episode Length During Training')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Steps per Episode')
axes[1].legend()

fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 10.4 Save trained model
# ---------------------------------------------------------------------------
MODEL_SAVE_PATH = "dqn_nifty50_generic_agent"
model.save(MODEL_SAVE_PATH)

model_file = MODEL_SAVE_PATH + ".zip"
if os.path.exists(model_file):
    model_size_mb = os.path.getsize(model_file) / (1024 * 1024)
    print(f"Model saved to: {model_file}")
    print(f"Model file size: {model_size_mb:.2f} MB")
else:
    print(f"Model saved to: {MODEL_SAVE_PATH}")

---

# 11. Evaluation on Test Set

The trained agent is evaluated on the held-out test set with the policy frozen
(deterministic=True, no exploration). Performance is measured per stock and
aggregated across the full NIFTY 50 universe.

Metrics computed:
- **Sharpe Ratio:** risk-adjusted return, annualised with the standard x sqrt(252) factor
- **Maximum Drawdown:** largest peak-to-trough decline in the equity curve
- **Annualised Return:** compound annual growth rate over the test period
- **Win Rate:** fraction of trading days with positive returns
- **Total Return:** cumulative percentage return over the test period

In [ ]:
# ---------------------------------------------------------------------------
# 11.1 Evaluation metric functions
# ---------------------------------------------------------------------------
def calculate_sharpe_ratio(daily_returns, trading_days=TRADING_DAYS_PER_YEAR):
    """Compute annualised Sharpe Ratio."""
    mean_ret = np.mean(daily_returns)
    std_ret = np.std(daily_returns, ddof=1)
    if std_ret == 0 or np.isnan(std_ret):
        return 0.0
    return float((mean_ret / std_ret) * np.sqrt(trading_days))


def calculate_max_drawdown(equity_curve):
    """Compute maximum drawdown as a positive fraction."""
    if len(equity_curve) == 0:
        return 0.0
    running_max = np.maximum.accumulate(equity_curve)
    drawdowns = (running_max - equity_curve) / np.where(running_max > 0, running_max, 1.0)
    return float(np.max(drawdowns))


def calculate_annualised_return(final_value, initial_value, num_trading_days,
                                 trading_days_per_year=TRADING_DAYS_PER_YEAR):
    """Compute annualised return."""
    if initial_value <= 0 or num_trading_days <= 0:
        return 0.0
    return float(
        (final_value / initial_value) ** (trading_days_per_year / num_trading_days) - 1
    )


print("Evaluation metric functions defined.")

In [ ]:
# ---------------------------------------------------------------------------
# 11.2 Per-stock evaluation function
# ---------------------------------------------------------------------------
def evaluate_agent_on_stock(model, stock_data, feature_cols, stock_symbol=""):
    """
    Run the frozen agent through a full stock test period and compute all metrics.

    Returns
    -------
    dict
        Dictionary of evaluation metrics and arrays for plotting.
    """
    env = NiftyTradingEnvFixed(
        stock_data=stock_data,
        feature_cols=feature_cols,
        random_start=False,
    )

    obs, _ = env.reset()
    daily_returns = []
    portfolio_values = [INITIAL_PORTFOLIO_VALUE]
    actions_taken = []
    positions = []
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        action = int(action)
        obs, reward, terminated, truncated, info = env.step(action)

        pos = info.get('position', 0)
        dr = info.get('daily_return', 0.0) * pos
        daily_returns.append(dr)
        portfolio_values.append(info.get('portfolio_value', portfolio_values[-1]))
        actions_taken.append(action)
        positions.append(pos)
        done = terminated or truncated

    daily_returns = np.array(daily_returns)
    portfolio_values = np.array(portfolio_values)
    actions_taken = np.array(actions_taken)

    # Compute metrics
    sharpe = calculate_sharpe_ratio(daily_returns)
    max_dd = calculate_max_drawdown(portfolio_values)
    ann_ret = calculate_annualised_return(
        portfolio_values[-1], portfolio_values[0], len(daily_returns)
    )
    total_ret = (portfolio_values[-1] / portfolio_values[0] - 1) * 100
    win_rate = np.mean(daily_returns > 0) * 100 if len(daily_returns) > 0 else 0.0

    # Action distribution
    action_counts = {0: 0, 1: 0, 2: 0}
    for a in actions_taken:
        action_counts[a] = action_counts.get(a, 0) + 1

    return {
        'symbol': stock_symbol,
        'sharpe_ratio': sharpe,
        'max_drawdown': max_dd,
        'annualised_return': ann_ret,
        'total_return_pct': total_ret,
        'win_rate_pct': win_rate,
        'num_trades': sum(1 for i in range(1, len(positions))
                          if positions[i] != positions[i-1]),
        'buy_count': action_counts[0],
        'hold_count': action_counts[1],
        'sell_count': action_counts[2],
        'daily_returns': daily_returns,
        'portfolio_values': portfolio_values,
        'actions': actions_taken,
        'positions': positions,
    }


print("Per-stock evaluation function defined.")

In [ ]:
# ---------------------------------------------------------------------------
# 11.3 Run evaluation on all test stocks
# ---------------------------------------------------------------------------
print("=" * 70)
print("Evaluating DQN agent on test set (all stocks, deterministic mode)")
print("=" * 70)

# Start test performance measurement
tracemalloc.start()
test_start_time = time.perf_counter()
test_mem_snapshot_before = tracemalloc.take_snapshot()

# Load the saved model
loaded_model = DQN.load(MODEL_SAVE_PATH)

eval_results = []
eval_details = {}

for i, (symbol, stock_data) in enumerate(sorted(test_stock_dict.items())):
    result = evaluate_agent_on_stock(loaded_model, stock_data, STATE_FEATURES, symbol)
    eval_results.append(result)
    eval_details[symbol] = result

    if (i + 1) % 10 == 0 or (i + 1) == len(test_stock_dict):
        print(f"  Evaluated {i+1}/{len(test_stock_dict)} stocks...")

# Stop test performance measurement
test_end_time = time.perf_counter()
test_mem_snapshot_after = tracemalloc.take_snapshot()
test_wall_time = test_end_time - test_start_time

test_mem_stats = test_mem_snapshot_after.compare_to(test_mem_snapshot_before, 'lineno')
test_peak_memory_bytes = sum(stat.size_diff for stat in test_mem_stats if stat.size_diff > 0)
test_peak_memory_mb = test_peak_memory_bytes / (1024 * 1024)

tracemalloc.stop()

print(f"\nTest evaluation complete.")
print(f"  Wall-clock time: {test_wall_time:.2f} seconds")
print(f"  Memory delta: {test_peak_memory_mb:.2f} MB")

In [ ]:
# ---------------------------------------------------------------------------
# 11.4 Per-stock results table
# ---------------------------------------------------------------------------
results_table = pd.DataFrame([{
    'Symbol': r['symbol'],
    'Sharpe Ratio': r['sharpe_ratio'],
    'Max Drawdown': r['max_drawdown'],
    'Ann. Return': r['annualised_return'],
    'Total Return %': r['total_return_pct'],
    'Win Rate %': r['win_rate_pct'],
    'Num Trades': r['num_trades'],
} for r in eval_results])

results_table = results_table.sort_values('Sharpe Ratio', ascending=False)

print("\nDQN Agent - Per-Stock Test Results (sorted by Sharpe Ratio):")
print("=" * 90)
print(results_table.to_string(index=False, float_format='%.4f'))
print("=" * 90)

In [ ]:
# ---------------------------------------------------------------------------
# 11.5 Aggregate metrics
# ---------------------------------------------------------------------------
agg_metrics = {
    'Mean Sharpe Ratio': results_table['Sharpe Ratio'].mean(),
    'Median Sharpe Ratio': results_table['Sharpe Ratio'].median(),
    'Mean Max Drawdown': results_table['Max Drawdown'].mean(),
    'Mean Annualised Return': results_table['Ann. Return'].mean(),
    'Mean Total Return %': results_table['Total Return %'].mean(),
    'Mean Win Rate %': results_table['Win Rate %'].mean(),
    'Stocks with Positive Sharpe': (results_table['Sharpe Ratio'] > 0).sum(),
    'Total Stocks Evaluated': len(results_table),
}

print("\nAggregate DQN Performance Across All Stocks:")
print("-" * 50)
for metric, value in agg_metrics.items():
    if isinstance(value, float):
        print(f"  {metric:.<35s} {value:.4f}")
    else:
        print(f"  {metric:.<35s} {value}")

---

# 12. Buy-and-Hold Baseline Comparison

The Buy-and-Hold strategy represents the simplest possible investment approach:
purchase the stock on the first day of the test period and hold it until the end.
This serves as the primary benchmark that the DQN agent must outperform to justify
the added complexity of a reinforcement learning approach.

For each stock, we compute the same metrics (Sharpe Ratio, Maximum Drawdown,
Annualised Return) on the Buy-and-Hold returns and compare them directly against
the DQN agent's performance.

In [ ]:
# ---------------------------------------------------------------------------
# 12.1 Compute Buy-and-Hold metrics for each test stock
# ---------------------------------------------------------------------------
bh_results = []

for symbol, stock_data in sorted(test_stock_dict.items()):
    stock_df = stock_data.sort_values('Date').reset_index(drop=True)

    if 'Raw_Close' in stock_df.columns:
        closes = stock_df['Raw_Close'].values
    else:
        closes = stock_df['Close'].values

    if len(closes) < 2:
        continue

    # Buy-and-Hold daily returns
    bh_daily_returns = np.diff(closes) / closes[:-1]

    # Equity curve
    bh_equity = INITIAL_PORTFOLIO_VALUE * np.cumprod(1 + bh_daily_returns)
    bh_equity = np.insert(bh_equity, 0, INITIAL_PORTFOLIO_VALUE)

    # Metrics
    sharpe = calculate_sharpe_ratio(bh_daily_returns)
    max_dd = calculate_max_drawdown(bh_equity)
    ann_ret = calculate_annualised_return(
        bh_equity[-1], bh_equity[0], len(bh_daily_returns)
    )
    total_ret = (bh_equity[-1] / bh_equity[0] - 1) * 100

    bh_results.append({
        'Symbol': symbol,
        'BH_Sharpe': sharpe,
        'BH_MaxDD': max_dd,
        'BH_AnnReturn': ann_ret,
        'BH_TotalReturn': total_ret,
    })

bh_df = pd.DataFrame(bh_results)
print(f"Buy-and-Hold baseline computed for {len(bh_df)} stocks.")

In [ ]:
# ---------------------------------------------------------------------------
# 12.2 Comparison table: DQN vs Buy-and-Hold
# ---------------------------------------------------------------------------
comparison_df = results_table[['Symbol', 'Sharpe Ratio', 'Max Drawdown',
                                'Ann. Return', 'Total Return %']].copy()
comparison_df = comparison_df.merge(bh_df, on='Symbol', how='inner')

comparison_df['Sharpe_Diff'] = comparison_df['Sharpe Ratio'] - comparison_df['BH_Sharpe']
comparison_df['DD_Diff'] = comparison_df['Max Drawdown'] - comparison_df['BH_MaxDD']

dqn_wins_sharpe = (comparison_df['Sharpe_Diff'] > 0).sum()
total_stocks = len(comparison_df)

print("\nDQN vs Buy-and-Hold Comparison:")
print("=" * 100)
print(comparison_df.to_string(index=False, float_format='%.4f'))
print("=" * 100)
print(f"\nDQN outperforms Buy-and-Hold on Sharpe Ratio: "
      f"{dqn_wins_sharpe}/{total_stocks} stocks ({dqn_wins_sharpe/total_stocks*100:.1f}%)")

In [ ]:
# ---------------------------------------------------------------------------
# 12.3 Comparison bar charts
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Select top and bottom 10 stocks by Sharpe difference for readability
comp_sorted = comparison_df.sort_values('Sharpe_Diff', ascending=False)
n_show = min(15, len(comp_sorted))
comp_subset = pd.concat([comp_sorted.head(n_show // 2 + 1),
                          comp_sorted.tail(n_show // 2)])

x = np.arange(len(comp_subset))
width = 0.35

# Sharpe Ratio
axes[0].barh(x - width/2, comp_subset['Sharpe Ratio'].values, width,
             label='DQN', color=PALETTE[0], edgecolor='none')
axes[0].barh(x + width/2, comp_subset['BH_Sharpe'].values, width,
             label='Buy-and-Hold', color=PALETTE[1], edgecolor='none')
axes[0].set_yticks(x)
axes[0].set_yticklabels(comp_subset['Symbol'].values, fontsize=8)
axes[0].set_title('Sharpe Ratio Comparison')
axes[0].legend()
axes[0].axvline(0, color='black', linewidth=0.8)

# Max Drawdown
axes[1].barh(x - width/2, comp_subset['Max Drawdown'].values, width,
             label='DQN', color=PALETTE[0], edgecolor='none')
axes[1].barh(x + width/2, comp_subset['BH_MaxDD'].values, width,
             label='Buy-and-Hold', color=PALETTE[1], edgecolor='none')
axes[1].set_yticks(x)
axes[1].set_yticklabels(comp_subset['Symbol'].values, fontsize=8)
axes[1].set_title('Max Drawdown Comparison (lower is better)')
axes[1].legend()

# Annualised Return
axes[2].barh(x - width/2, comp_subset['Ann. Return'].values * 100, width,
             label='DQN', color=PALETTE[0], edgecolor='none')
axes[2].barh(x + width/2, comp_subset['BH_AnnReturn'].values * 100, width,
             label='Buy-and-Hold', color=PALETTE[1], edgecolor='none')
axes[2].set_yticks(x)
axes[2].set_yticklabels(comp_subset['Symbol'].values, fontsize=8)
axes[2].set_title('Annualised Return (%) Comparison')
axes[2].legend()
axes[2].axvline(0, color='black', linewidth=0.8)

fig.suptitle('DQN Agent vs Buy-and-Hold Baseline', fontsize=14)
fig.tight_layout()
plt.show()

---

# 13. Action Distribution Analysis

Inspecting the distribution of actions taken by the agent is critical for detecting
degenerate policies. A well-functioning agent should exhibit a meaningful mix of Buy,
Hold, and Sell actions. An agent that overwhelmingly selects a single action (e.g.,
"always Hold") has collapsed into a trivial policy and is not genuinely learning to
trade.

This section examines both the aggregate action distribution and the temporal pattern
of positions for representative stocks.

In [ ]:
# ---------------------------------------------------------------------------
# 13.1 Aggregate action distribution
# ---------------------------------------------------------------------------
total_actions = {0: 0, 1: 0, 2: 0}
for result in eval_results:
    total_actions[0] += result['buy_count']
    total_actions[1] += result['hold_count']
    total_actions[2] += result['sell_count']

action_labels = ['Buy', 'Hold', 'Sell']
action_values = [total_actions[0], total_actions[1], total_actions[2]]
total = sum(action_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors_pie = [PALETTE[0], PALETTE[2], PALETTE[3]]
wedges, texts, autotexts = axes[0].pie(
    action_values, labels=action_labels, autopct='%1.1f%%',
    colors=colors_pie, startangle=90,
    textprops={'fontsize': 11}
)
axes[0].set_title('Aggregate Action Distribution (All Test Stocks)')

# Per-stock action mix (stacked bar for representative stocks)
sample_results = eval_results[:min(10, len(eval_results))]
symbols = [r['symbol'] for r in sample_results]
buy_pcts = [r['buy_count'] / max(1, r['buy_count'] + r['hold_count'] + r['sell_count']) * 100
            for r in sample_results]
hold_pcts = [r['hold_count'] / max(1, r['buy_count'] + r['hold_count'] + r['sell_count']) * 100
             for r in sample_results]
sell_pcts = [r['sell_count'] / max(1, r['buy_count'] + r['hold_count'] + r['sell_count']) * 100
             for r in sample_results]

y_pos = np.arange(len(symbols))
axes[1].barh(y_pos, buy_pcts, color=PALETTE[0], label='Buy', edgecolor='none')
axes[1].barh(y_pos, hold_pcts, left=buy_pcts, color=PALETTE[2], label='Hold', edgecolor='none')
left_sell = [b + h for b, h in zip(buy_pcts, hold_pcts)]
axes[1].barh(y_pos, sell_pcts, left=left_sell, color=PALETTE[3], label='Sell', edgecolor='none')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(symbols, fontsize=9)
axes[1].set_xlabel('Percentage of Actions')
axes[1].set_title('Action Distribution per Stock (Sample)')
axes[1].legend(loc='lower right')

fig.tight_layout()
plt.show()

print(f"\nAggregate action counts: Buy={total_actions[0]:,}, "
      f"Hold={total_actions[1]:,}, Sell={total_actions[2]:,}")
print(f"Total actions: {total:,}")

In [ ]:
# ---------------------------------------------------------------------------
# 13.2 Position overlay on stock price (representative stock)
# ---------------------------------------------------------------------------
# Pick the stock with the best Sharpe Ratio for a meaningful visualisation
best_stock = results_table.iloc[0]['Symbol']
if best_stock in eval_details:
    detail = eval_details[best_stock]
    stock_test_data = test_stock_dict[best_stock].sort_values('Date').reset_index(drop=True)

    fig, ax1 = plt.subplots(figsize=(14, 6))

    # Plot price
    dates = stock_test_data['Date'].values[:len(detail['positions'])]
    if 'Raw_Close' in stock_test_data.columns:
        prices = stock_test_data['Raw_Close'].values[:len(detail['positions'])]
    else:
        prices = stock_test_data['Close'].values[:len(detail['positions'])]

    ax1.plot(dates, prices, color=PALETTE[0], linewidth=1, label='Close Price')
    ax1.set_ylabel('Close Price (INR)', color=PALETTE[0])
    ax1.set_xlabel('Date')

    # Overlay positions
    ax2 = ax1.twinx()
    positions = np.array(detail['positions'])
    ax2.fill_between(dates, 0, positions, alpha=0.2, color=PALETTE[1],
                     step='post', label='Position (1=Long)')
    ax2.set_ylabel('Position', color=PALETTE[1])
    ax2.set_ylim(-0.1, 1.5)

    ax1.set_title(f'Agent Trading Behaviour - {best_stock} (Test Period)')
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    fig.tight_layout()
    plt.show()
else:
    print(f"No detailed results available for {best_stock}")

---

# 14. Performance Measurement Summary

This section consolidates the computational resource consumption of the training and
testing phases, including wall-clock execution time, memory footprint, and saved model
size on disk.

In [ ]:
# ---------------------------------------------------------------------------
# 14.1 Formatted performance summary
# ---------------------------------------------------------------------------
print("=" * 60)
print("COMPUTATIONAL PERFORMANCE SUMMARY")
print("=" * 60)

print(f"\n{'Phase':<25s} {'Wall Time':>15s} {'Memory Delta':>15s}")
print("-" * 60)
print(f"{'Training':<25s} {train_wall_time:>12.2f} s  {train_peak_memory_mb:>12.2f} MB")
print(f"{'Testing (all stocks)':<25s} {test_wall_time:>12.2f} s  {test_peak_memory_mb:>12.2f} MB")
print("-" * 60)
print(f"{'Total':<25s} {train_wall_time + test_wall_time:>12.2f} s  "
      f"{train_peak_memory_mb + test_peak_memory_mb:>12.2f} MB")

# Model file size
model_file_path = MODEL_SAVE_PATH + ".zip"
if os.path.exists(model_file_path):
    model_mb = os.path.getsize(model_file_path) / (1024 * 1024)
else:
    model_mb = 0.0

print(f"\nModel file size on disk: {model_mb:.2f} MB")
print(f"Training timesteps: {TRAIN_TIMESTEPS:,}")
print(f"Training episodes completed: {len(callback.episode_rewards)}")
print(f"Throughput: {TRAIN_TIMESTEPS / max(train_wall_time, 0.01):,.0f} timesteps/second")
print("=" * 60)

---

# 15. Model Export and Inference Wrapper

The saved model file (a .zip archive containing network weights, architecture
definition, and hyperparameters) can be reloaded for inference without retraining.

Below is a clean inference wrapper function designed for integration with
Subproblem D's FastAPI deployment backend.

In [ ]:
# ---------------------------------------------------------------------------
# 15.1 Inference wrapper function
# ---------------------------------------------------------------------------
def predict_action(ohlcv_window, position, model_path=MODEL_SAVE_PATH):
    """
    Predict a trading action given a feature window and current position.

    Parameters
    ----------
    ohlcv_window : np.ndarray
        Shape (window_size, num_features) array of normalised feature values.
    position : int
        Current position: 0 (flat) or 1 (long).
    model_path : str
        Path to the saved DQN model (without .zip extension).

    Returns
    -------
    str
        One of 'Buy', 'Hold', or 'Sell'.
    """
    action_names = {0: 'Buy', 1: 'Hold', 2: 'Sell'}

    # Load model
    loaded = DQN.load(model_path)

    # Build observation vector
    flat_window = ohlcv_window.flatten().astype(np.float32)
    position_feature = np.array([position], dtype=np.float32)
    obs = np.concatenate([flat_window, position_feature])

    # Predict
    action, _ = loaded.predict(obs, deterministic=True)
    return action_names[int(action)]


# Demonstration
print("Inference wrapper function defined.")
print("\nDemonstration with random input:")
demo_window = np.random.randn(WINDOW_SIZE, NUM_FEATURES).astype(np.float32)
demo_action = predict_action(demo_window, position=0)
print(f"  Input shape: ({WINDOW_SIZE}, {NUM_FEATURES})")
print(f"  Position: 0 (flat)")
print(f"  Predicted action: {demo_action}")

---

# 16. Conclusions

## Summary of Findings

This notebook successfully implemented and evaluated a Deep Q-Network reinforcement
learning agent for algorithmic trading on the NIFTY 50 stock universe. The key
outcomes are:

1. A single generic DQN agent was trained across all available NIFTY 50 constituent
   stocks, demonstrating that normalised feature representations enable cross-stock
   transfer of learned trading patterns.

2. Hyperparameters were selected through a programmatic search over the validation
   set, avoiding manual tuning and ensuring reproducibility.

3. The agent's action distribution was analysed to verify it learned a non-trivial
   policy rather than collapsing into a degenerate "always Hold" strategy.

4. Performance was compared against a passive Buy-and-Hold baseline using Sharpe
   Ratio, Maximum Drawdown, and Annualised Return.

## Limitations

1. **Non-stationarity.** The dataset spans multiple distinct market regimes (2000-2021),
   including the dot-com aftermath, the 2008 global financial crisis, and the 2020
   COVID-19 crash. A single agent trained across all regimes may converge to an
   "average regime" policy that performs suboptimally in any specific regime.

2. **Reward calibration.** The reward function uses a fixed 0.1% transaction cost and
   simple position-weighted daily returns. More sophisticated reward shaping (e.g.,
   risk-adjusted rewards or drawdown penalties) could improve the agent's behaviour.

3. **No sentiment features.** The current state vector does not include news sentiment
   signals. Integration of Subproblem D's sentiment pipeline may improve performance,
   particularly around event-driven price movements.

4. **Single-position constraint.** The agent can only be flat or long (no short
   selling), which limits its ability to profit from downward price movements.

5. **Training scale.** Kaggle compute constraints limit the number of training
   timesteps. A more extended training run on dedicated hardware could improve
   convergence.

## Future Work

- **Sentiment integration (Subproblem D):** Expand the state vector with sentiment
  features to capture news-driven market dynamics.
- **XGBoost comparison (Subproblem A):** Add Nitesh's supervised baseline to the
  comparison table for a complete evaluation of RL vs supervised approaches.
- **Advanced DQN variants:** Evaluate Double DQN, Duelling DQN, or Prioritised
  Experience Replay for potential stability and performance improvements.
- **Multi-regime analysis:** Evaluate the agent separately on bull, bear, and
  sideways market regimes to understand regime-dependent performance.

---

**Notebook execution complete.**